# WordNet

**ID** — Dokumen `.docx`: paragraf, style, tabel, section, mail merge, ekspor PDF.
**EN** — `.docx` documents: paragraphs, styles, tables, sections, mail merge, PDF export.

> Dibuat oleh Gravicode Studios, dipimpin oleh Kang Fadhil.

Panduan lengkap / full guide: [`docs/WordNet.md`](../docs/WordNet.md) ·
[Bahasa Indonesia](../docs/id/WordNet.md)

In [ ]:
// Build first:  dotnet build OfficeNet.sln -c Release

#r "../src/OfficeNet.Core/bin/Release/net10.0/Gravicode.OfficeNet.Core.dll"
#r "../src/PdfNet/bin/Release/net10.0/Gravicode.OfficeNet.PdfNet.dll"
#r "../src/WordNet/bin/Release/net10.0/Gravicode.OfficeNet.WordNet.dll"
#r "../src/OfficeNet.Rendering/bin/Release/net10.0/Gravicode.OfficeNet.Rendering.dll"

// Published instead? Swap the lines above for:
//   #r "nuget: Gravicode.OfficeNet, *"

In [ ]:
using OfficeNet.Rendering;
using Microsoft.DotNet.Interactive.Formatting;

// Renders a document and shows the first page inline, so a cell's effect is visible rather than
// described. Base64 in an <img> because the notebook has nowhere to serve a file from.
void Show(string path, int width = 520)
{
    var png = DocumentRenderer.RenderThumbnail(path, width);
    var data = Convert.ToBase64String(png);

    display(HTML($"<img src='data:image/png;base64,{data}' style='border:1px solid #ddd' />"));
}

var work = Path.Combine(Path.GetTempPath(), "officenet-notebook");
Directory.CreateDirectory(work);
string At(string name) => Path.Combine(work, name);

Console.WriteLine($"Berkas ditulis ke / files written to: {work}");

## Paragraf dan run / Paragraphs and runs

Format bersifat tiga keadaan: `null` mewarisi, `false` mematikan secara eksplisit. Itulah satu-satunya
cara menulis kata tidak tebal di dalam heading tebal. /
Formatting is tri-state: `null` inherits, `false` is explicitly off. That is the only way to write an
unbolded word inside a bold heading.

In [ ]:
using WordNet;
using WordNet.Styles;
using OfficeNet.Core;
using OfficeNet.Core.Drawing;

var document = WordDocument.Create();

var paragraph = document.AddParagraph();
paragraph.Alignment = ParagraphAlignment.Justify;
paragraph.AddRun("Pendapatan tumbuh ");
paragraph.AddRun("32%", bold: true);
paragraph.AddRun(" dibanding tahun sebelumnya, ditopang permintaan yang kuat di Jakarta.");

var emphasis = document.AddParagraph().AddRun("Miring, bergaris bawah, berwarna.");
emphasis.Format.Italic = true;
emphasis.Format.Underline = UnderlineStyle.Single;
emphasis.Format.Color = OfficeColor.FromRgb(0x1F, 0x38, 0x64);
emphasis.Format.FontSize = Units.Pt(13);

document.ExtractText()

## Heading, daftar, style / Headings, lists, styles

In [ ]:
document.AddHeading("Komponen", 1);

document.AddList([
    "WordNet — python-docx",
    "ExcelNet — openpyxl + pandas",
    "PowerPointNet — python-pptx",
    "PdfNet — PyPDF2",
]);

document.AddHeading("Langkah", 1);
document.AddList(["Pasang paket", "Tulis kode", "Simpan"], numbered: true);

var style = document.Styles.GetOrAdd("Catatan", "Catatan", StyleType.Paragraph, basedOn: "Normal");
style.RunFormat.FontSize = Units.Pt(9);
style.RunFormat.Italic = true;

document.AddParagraph("Angka bersifat ilustratif.", "Catatan");

document.Styles.All.Count()

## Tabel / Tables

`SetColumnWidth` juga mengalihkan tabel ke layout tetap — tanpa itu Word menghitung ulang setiap
kolom dan lebar yang Anda set diabaikan. /
`SetColumnWidth` also switches the table to fixed layout — without it Word recomputes every column
and your widths are ignored.

In [ ]:
document.AddHeading("Pendapatan per Wilayah", 1);

var table = document.AddTable(new[]
{
    new[] { "Wilayah", "2025", "2026", "Pertumbuhan" },
    new[] { "Jakarta", "1.120", "1.480", "+32%" },
    new[] { "Bandung", "860", "1.150", "+34%" },
    new[] { "Surabaya", "740", "905", "+22%" },
});

table.Rows[0].SetShading(OfficeColor.FromRgb(0x1F, 0x38, 0x64));

foreach (var cell in table.Rows[0].Cells)
    foreach (var run in cell.Paragraphs.SelectMany(p => p.Runs))
        run.Format.Color = OfficeColor.White;

$"{table.Count} baris x {table.ColumnCount} kolom"

## Header, footer, dan nomor halaman / Headers, footers and page numbers

Nomor halaman adalah *field*. Cache-nya pada dokumen buatan program selalu basi — pengekspor PDF
mengisinya per halaman. /
Page numbers are *fields*. A generated document's cache is always stale — the PDF exporter fills in
the real numbers per page.

In [ ]:
var section = document.Section;
section.SetPageSize("A4");
section.SetMargins(Units.Cm(2.2));

section.GetHeader().AddParagraph("Gravicode Studios").Alignment = ParagraphAlignment.Right;

var footer = section.GetFooter().AddParagraph();
footer.Alignment = ParagraphAlignment.Center;
footer.AddRun("Halaman ");
footer.AddPageNumber();
footer.AddRun(" dari ");
footer.AddPageCount();

document.Properties.Title = "Laporan Tahunan";
document.Properties.Creator = "Gravicode Studios";

document.Save(At("wordnet.docx"));
Show(At("wordnet.docx"));

## Mail merge dan penggantian teks / Mail merge and replacement

Word memecah teks antar-run di titik sembarang, jadi frasa yang Anda lihat sering tidak ada di satu
run pun — itulah gunanya `ReplaceTextAcrossRuns`. /
Word splits text across runs at arbitrary points, so a phrase you can see is often in no single
run — that is what `ReplaceTextAcrossRuns` is for.

In [ ]:
using var letter = WordDocument.Create();
letter.AddParagraph("Kepada {{nama}} di {{kota}},");
letter.AddParagraph("Terima kasih atas pesanan Anda pada tahun 2025.");

letter.MailMerge(new Dictionary<string, string>
{
    ["nama"] = "Budi Santoso",
    ["kota"] = "Bandung",
});

letter.ReplaceTextAcrossRuns("2025", "2026");

letter.ExtractText()

## Ekspor PDF / PDF export

In [ ]:
document.SaveAsPdf(At("wordnet.pdf"), new WordNet.Export.PdfExportOptions
{
    Watermark = "DRAF",
    WatermarkOpacity = 0.10,
});

Show(At("wordnet.pdf"));

In [ ]:
document.Dispose();